In [1]:
# ======================================================================
# 07_figures.ipynb
# FinGap-Governance | Publication figures.
# ----------------------------------------------------------------------
# Reads results/tables/*_runs.json and metrics_*.csv, renders GREYSCALE
# seaborn figures with NO embedded captions, and saves each as BOTH .png
# and .pdf at dpi=600 under results/figures/. Self-contained.
# ======================================================================
from fingap_core import *
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white", context="paper")
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.edgecolor": "black",
                     "axes.linewidth": 0.8, "savefig.bbox": "tight"})
GREYS = sns.color_palette("Greys", n_colors=4)
COND_ORDER = ["direct", "vanilla_rag", "graph_rag"]
COND_LBL   = {"direct": "Direct", "vanilla_rag": "Vanilla RAG", "graph_rag": "GraphRAG"}
HATCH      = {"direct": "...", "vanilla_rag": "///", "graph_rag": ""}

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=600)
    plt.close(fig)
    print("saved", name + ".png/.pdf")

df = pd.concat([pd.read_json(TABLES / f"{t}_runs.json") for t in TASKS
                if (TABLES / f"{t}_runs.json").exists()], ignore_index=True)
assert len(df), "no run records -- run 03-05 and 06 first"
r = lambda s: 100 * s.mean()

# --- Fig A: traceability completion rate, task x condition (greyscale heatmap) ---
pivot = (df.groupby(["task", "condition"])["traceable"].mean().mul(100)
           .unstack().reindex(columns=COND_ORDER).reindex(index=TASKS))
fig, ax = plt.subplots(figsize=(5.0, 3.2))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Greys", vmin=0, vmax=100,
            linewidths=0.6, linecolor="black", cbar_kws={"label": "Traceability completion (%)"},
            ax=ax, annot_kws={"color": "black"})
ax.set_xticklabels([COND_LBL[c] for c in pivot.columns]); ax.set_ylabel(""); ax.set_xlabel("")
save(fig, "fig_traceability_matrix")

# --- Fig B: determination accuracy vs traceability, grouped bars per condition ---
acc = df.groupby("condition")["determination_correct"].apply(r).reindex(COND_ORDER)
trc = df.groupby("condition")["traceable"].apply(r).reindex(COND_ORDER)
x = np.arange(len(COND_ORDER)); w = 0.38
fig, ax = plt.subplots(figsize=(5.2, 3.3))
ax.bar(x - w/2, acc.values, w, color=GREYS[1], edgecolor="black", label="Determination accuracy")
ax.bar(x + w/2, trc.values, w, color=GREYS[3], edgecolor="black", hatch="///", label="Traceability completion")
ax.set_xticks(x); ax.set_xticklabels([COND_LBL[c] for c in COND_ORDER])
ax.set_ylabel("%"); ax.set_ylim(0, 100); ax.legend(frameon=False, fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
save(fig, "fig_accuracy_vs_traceability")

# --- Fig C: correct-but-untraceable rate, task x condition (the thesis) ---
cu = (df.groupby(["task", "condition"])["correct_but_untraceable"].mean().mul(100)
        .unstack().reindex(columns=COND_ORDER).reindex(index=TASKS))
fig, ax = plt.subplots(figsize=(5.4, 3.3))
xx = np.arange(len(TASKS)); w = 0.26
for i, c in enumerate(COND_ORDER):
    ax.bar(xx + (i-1)*w, cu[c].values, w, color=GREYS[i+1], edgecolor="black",
           hatch=HATCH[c], label=COND_LBL[c])
ax.set_xticks(xx); ax.set_xticklabels(TASKS); ax.set_ylabel("Correct but untraceable (%)")
ax.set_ylim(0, 100); ax.legend(frameon=False, fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
save(fig, "fig_correct_but_untraceable")

# --- Fig D: failure-type profile by condition (stacked greyscale bars) ---
prof = (df.dropna(subset=["failure"]).groupby(["condition", "failure"]).size()
          .unstack(fill_value=0).reindex(index=COND_ORDER)
          .reindex(columns=FAILURE_TYPES, fill_value=0))
prop = prof.div(prof.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(6.0, 3.4))
bottom = np.zeros(len(COND_ORDER))
greys5 = sns.color_palette("Greys", n_colors=len(FAILURE_TYPES) + 1)[1:]
for j, ft in enumerate(FAILURE_TYPES):
    ax.bar([COND_LBL[c] for c in COND_ORDER], prop[ft].values, bottom=bottom,
           color=greys5[j], edgecolor="black", linewidth=0.5, label=ft)
    bottom += prop[ft].values
ax.set_ylabel("Share of failures"); ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
for s in ("top", "right"): ax.spines[s].set_visible(False)
save(fig, "fig_failure_profile")

# --- Fig E: cross-task failure-profile similarity per condition (isomorphism) ---
iso = pd.read_csv(TABLES / "metrics_isomorphism.csv") if (TABLES/"metrics_isomorphism.csv").exists() else None
if iso is not None:
    piv = iso.pivot(index="task_pair", columns="condition", values="cosine_similarity").reindex(columns=COND_ORDER)
    fig, ax = plt.subplots(figsize=(5.0, 3.0))
    sns.heatmap(piv, annot=True, fmt=".2f", cmap="Greys", vmin=0, vmax=1,
                linewidths=0.6, linecolor="black", ax=ax,
                cbar_kws={"label": "Cosine similarity"}, annot_kws={"color": "black"})
    ax.set_xticklabels([COND_LBL[c] for c in piv.columns]); ax.set_xlabel(""); ax.set_ylabel("")
    save(fig, "fig_isomorphism_similarity")

print("\nall figures saved under results/figures/ (png + pdf, dpi 600)")


saved fig_traceability_matrix.png/.pdf
saved fig_accuracy_vs_traceability.png/.pdf
saved fig_correct_but_untraceable.png/.pdf
saved fig_failure_profile.png/.pdf
saved fig_isomorphism_similarity.png/.pdf

all figures saved under results/figures/ (png + pdf, dpi 600)
